# Выбор API по запросу и перенос с ToolLens на ToolBench

Сравниваем Popularity, BM25, BGE и три семейства моделей коллаборативной фильтрации. Основной критерий — доля запросов, для которых первые 10 рекомендаций содержат все размеченные API (FullCoverage@10, FC@10). Recall учитывает долю найденных API, NDCG — ещё и их позиции.

Запускать из корня репозитория или из папки `experiments/`. Нужны Python 3.11+, Jupyter, NumPy, pandas, PyArrow, SciPy, scikit-learn, PyTorch, sentence-transformers, matplotlib и Jinja2. Используются локальные веса BGE. Каждое выполнение ячейки настройки создаёт отдельную папку результатов.

In [2]:
import os
from pathlib import Path
import sys

# Git и joblib могут запускать дочерние процессы после токенизации.
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
from threadpoolctl import threadpool_limits

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rankers').is_dir())
sys.path.insert(0, str(ROOT))

from experiments.dataset_io import CachedEncoder, load_dataset
from experiments.evaluation import evaluate_suite
from experiments.model_suite import (family_specs, fixed_specs, resolve_device,
                                     select_family, train_suite)
from experiments.workflow import (file_digest, start_run, store_selection, test_catalogue,
                                  transfer_group, write_json)

SEED, BOOTSTRAP_SEED = 67, 68
DEVICE = 'auto'  # auto, cpu, cuda, mps
device = resolve_device(DEVICE)
print(f'Устройство BGE и Adam: {device}')
torch.set_num_threads(4)
thread_limit = threadpool_limits(limits=4)
WEIGHTS = ROOT / 'models/bge-small-en-v1.5'
TOOLLENS = ROOT / 'data/ToolLens/processed'
TOOLBENCH = ROOT / 'data/ToolBench/processed'
GROUPS = ('G1', 'G2', 'G3')
fixed = fixed_specs(SEED, device=device)
encoder = CachedEncoder.from_local(WEIGHTS, device=device)
run = start_run(ROOT, WEIGHTS, fixed, family_specs(SEED, device=device),
                SEED, BOOTSTRAP_SEED)
print(run)

Устройство BGE и Adam: mps
/Users/svatoslavpolonskiy/Codex/CFTSagent/output/toollens_transfer/20261121T184936Z_67bdb523


## 1. Данные и 12 фиксированных методов

Читаем готовые запросы, API и взаимодействия. Все назначения train, val и test берём из `requests.parquet`. Каждый запрос сохраняет свой split, включая близкие формулировки. Возможные пересечения близких формулировок ограничивают интерпретацию качества.

На train обучаются Popularity, BM25, BGE и девять конфигураций CF. Query-KNN переносит факторы ближайших обучающих запросов, Family CF агрегирует историю запросов по кластерам, Projected CF обучает проекцию эмбеддинга запроса. Каждая CF-модель использует MSE, BPR со случайными негативами или BPR с негативами среди наиболее далёких API. Во время оценки кандидатами служат все API каталога.

In [5]:
source = load_dataset(TOOLLENS)
history = {api for links in source.qrels['train'].values() for api in links}
write_json(run / 'toollens_data.json', {name: file_digest(TOOLLENS / name) for name in
           ('apis.parquet', 'requests.parquet', 'interactions.parquet')})
models, fit_times = train_suite(source, encoder, fixed)
fit_times.to_csv(run / 'fixed_fits.csv', index=False)
fit_times[['model', 'fit_seconds']]

,model,fit_seconds
0,Popularity,0.020058
1,BM25,0.031792
2,BGE,10.389212
3,Query-KNN CF | MSE,36.923246
4,Query-KNN CF | BPR/random,4.964783
5,Query-KNN CF | BPR/semantic,6.099773
6,Family CF | MSE,1.685314
7,Family CF | BPR/random,1.786893
8,Family CF | BPR/semantic,2.004300
9,Projected CF | MSE,1.810564


## 2. Выбор Family CF на ToolLens validation

В сетке четыре способа кластеризации (MiniBatchKMeans с 32, 64 или 128 кластерами и DBSCAN), четыре способа агрегации (binary, count, sqrt, log1p) и три варианта обучения. Для каждой из 48 комбинаций проверяем 1, 2, 3, 5 и 10 соседей. Для неприменимых комбинаций сохраняется причина.

Первые три конфигурации выбираются по FullCoverage@10, затем Recall@10, NDCG@10 и имени. Выбранные варианты повторно обучаются на том же train. Фиксированные модели также оцениваем на validation, чтобы выбрать по одному представителю Query-KNN и Projected CF для итогового графика.

In [6]:
validation_dir = run / 'ToolLens' / 'validation'
family = select_family(source, encoder, seed=SEED, device=device,
                       artifact_dir=validation_dir / 'per_query')
family_val_table = store_selection(family, validation_dir)
fixed_validation = evaluate_suite(models, source, 'val', validation_dir / 'comparison',
                                  history, bootstrap_seed=BOOTSTRAP_SEED)
fixed_validation.to_csv(validation_dir / 'comparison' / 'curves.csv', index=False)
final_models = {**models, **family['models']}
family['scores'].head(3)

metric,model,FullCoverage,NDCG,Recall
78,Family CF | MiniBatchKMeans-128 | count | BPR/...,0.216303,0.487819,0.570059
76,Family CF | MiniBatchKMeans-128 | count | BPR/...,0.215770,0.494873,0.574853
79,Family CF | MiniBatchKMeans-128 | count | BPR/...,0.214172,0.491335,0.570769


## 3. Итог на ToolLens test

На одном наборе запросов сравниваем 12 фиксированных методов и три варианта Family CF, выбранных на validation. Отдельно оцениваем замороженную сетку Family. Нулевое число позитивов исключает запрос из средних и отражается в отчёте о покрытии. Для основного сравнения сохраняем парные 95% bootstrap-интервалы по запросам и разности с BGE.

In [7]:
toollens_table = test_catalogue(final_models, family['frozen_grid'], source,
                               run / 'ToolLens' / 'test', history, encoder, BOOTSTRAP_SEED,
                               context='ToolLens test')
toollens_table.round(4)

metric                                              Recall                  \
k                                                       3       5       10   
model                                                                        
Popularity                                          0.1439  0.1763  0.2399   
BM25                                                0.2469  0.2992  0.3606   
BGE                                                 0.2797  0.3478  0.4439   
Query-KNN CF | MSE                                  0.3809  0.4404  0.5273   
Query-KNN CF | BPR/random                           0.6731  0.7872  0.8761   
Query-KNN CF | BPR/semantic                         0.4273  0.5375  0.6708   
Family CF | MSE                                     0.2192  0.2686  0.3979   
Family CF | BPR/random                              0.1236  0.1736  0.2874   
Family CF | BPR/semantic                            0.0875  0.1302  0.2168   
Projected CF | MSE                                  0.3591  0.4103  0.5053   
Projected CF | BPR/random                           0.4664  0.6027  0.8030   
Projected CF | BPR/semantic                         0.2286  0.3038  0.4027   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.3563  0.4446  0.5685   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.3667  0.4492  0.5741   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.3622  0.4475  0.5724   

metric                                                        NDCG          \
k                                                       20      3       5    
model                                                                        
Popularity                                          0.3310  0.1611  0.1791   
BM25                                                0.4343  0.2695  0.2980   
BGE                                                 0.5515  0.2945  0.3310   
Query-KNN CF | MSE                                  0.6731  0.4372  0.4712   
Query-KNN CF | BPR/random                           0.9143  0.7038  0.7688   
Query-KNN CF | BPR/semantic                         0.7980  0.4479  0.5082   
Family CF | MSE                                     0.5561  0.2211  0.2489   
Family CF | BPR/random                              0.4258  0.1304  0.1587   
Family CF | BPR/semantic                            0.3180  0.0896  0.1137   
Projected CF | MSE                                  0.6129  0.4091  0.4384   
Projected CF | BPR/random                           0.9227  0.4849  0.5609   
Projected CF | BPR/semantic                         0.5232  0.2430  0.2853   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.6913  0.3861  0.4350   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.6929  0.3970  0.4430   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.6938  0.3921  0.4396   

metric                                                              \
k                                                       10      20   
model                                                                
Popularity                                          0.2077  0.2390   
BM25                                                0.3245  0.3493   
BGE                                                 0.3720  0.4090   
Query-KNN CF | MSE                                  0.5098  0.5597   
Query-KNN CF | BPR/random                           0.8091  0.8223   
Query-KNN CF | BPR/semantic                         0.5656  0.6092   
Family CF | MSE                                     0.3061  0.3595   
Family CF | BPR/random                              0.2094  0.2571   
Family CF | BPR/semantic                            0.1513  0.1865   
Projected CF | MSE                                  0.4803  0.5171   
Projected CF | BPR/random                           0.6475  0.6882   
Projected CF | BPR/semantic                         0.3290  0.3689   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.4899  0.5315   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.4983  0.5386   


## 4. Перенос на ToolBench G1, G2 и G3

Каждая группа использует свои готовые test-запросы и полный каталог. Обученные параметры и состав тройки Family зафиксированы по ToolLens. API сопоставляются по однозначному сочетанию имён инструмента и API или точному описанию. Новые и неоднозначные API получают отдельные ID. При малом пересечении каталогов результаты моделей с историей зависят от порядка разрешения равных оценок. Сопоставление и доля покрытых позитивов сохраняются для каждой группы.

Если каталог группы отсутствует в рабочем дереве, загрузчик читает подготовленные таблицы из локальной Git-ветки `main`. Для каждой группы сохраняется источник данных.

In [8]:
transfer_tables = {}
for group in GROUPS:
    transfer_tables[group] = transfer_group(
        ROOT, TOOLBENCH, group, source, final_models, family['frozen_grid'],
        history, encoder, run, BOOTSTRAP_SEED)
    display(transfer_tables[group].round(4))

metric                                              Recall                  \
k                                                       3       5       10   
model                                                                        
Popularity                                          0.0008  0.0009  0.0012   
BM25                                                0.3000  0.3596  0.4315   
BGE                                                 0.3470  0.4270  0.5277   
Query-KNN CF | MSE                                  0.0008  0.0009  0.0012   
Query-KNN CF | BPR/random                           0.0008  0.0009  0.0012   
Query-KNN CF | BPR/semantic                         0.0008  0.0009  0.0012   
Family CF | MSE                                     0.0008  0.0009  0.0012   
Family CF | BPR/random                              0.0008  0.0009  0.0012   
Family CF | BPR/semantic                            0.0008  0.0009  0.0012   
Projected CF | MSE                                  0.0008  0.0009  0.0012   
Projected CF | BPR/random                           0.0008  0.0009  0.0012   
Projected CF | BPR/semantic                         0.0008  0.0009  0.0012   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0008  0.0009  0.0012   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0008  0.0009  0.0012   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0008  0.0009  0.0012   

metric                                                        NDCG          \
k                                                       20      3       5    
model                                                                        
Popularity                                          0.0023  0.0009  0.0010   
BM25                                                0.4922  0.4352  0.4208   
BGE                                                 0.6151  0.4835  0.4819   
Query-KNN CF | MSE                                  0.0023  0.0009  0.0010   
Query-KNN CF | BPR/random                           0.0023  0.0009  0.0010   
Query-KNN CF | BPR/semantic                         0.0023  0.0009  0.0010   
Family CF | MSE                                     0.0023  0.0009  0.0010   
Family CF | BPR/random                              0.0023  0.0009  0.0010   
Family CF | BPR/semantic                            0.0023  0.0009  0.0010   
Projected CF | MSE                                  0.0023  0.0009  0.0010   
Projected CF | BPR/random                           0.0023  0.0009  0.0010   
Projected CF | BPR/semantic                         0.0023  0.0009  0.0010   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0023  0.0009  0.0010   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0023  0.0009  0.0010   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0023  0.0009  0.0010   

metric                                                              \
k                                                       10      20   
model                                                                
Popularity                                          0.0011  0.0015   
BM25                                                0.4216  0.4467   
BGE                                                 0.4957  0.5328   
Query-KNN CF | MSE                                  0.0011  0.0015   
Query-KNN CF | BPR/random                           0.0011  0.0015   
Query-KNN CF | BPR/semantic                         0.0011  0.0015   
Family CF | MSE                                     0.0011  0.0015   
Family CF | BPR/random                              0.0011  0.0015   
Family CF | BPR/semantic                            0.0011  0.0015   
Projected CF | MSE                                  0.0011  0.0015   
Projected CF | BPR/random                           0.0011  0.0015   
Projected CF | BPR/semantic                         0.0011  0.0015   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0011  0.0015   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0011  0.0015   


metric                                              Recall                  \
k                                                       3       5       10   
model                                                                        
Popularity                                          0.0003  0.0003  0.0014   
BM25                                                0.2649  0.3124  0.3769   
BGE                                                 0.2652  0.3153  0.3898   
Query-KNN CF | MSE                                  0.0026  0.0034  0.0063   
Query-KNN CF | BPR/random                           0.0080  0.0109  0.0153   
Query-KNN CF | BPR/semantic                         0.0045  0.0064  0.0111   
Family CF | MSE                                     0.0025  0.0034  0.0068   
Family CF | BPR/random                              0.0018  0.0029  0.0053   
Family CF | BPR/semantic                            0.0006  0.0012  0.0039   
Projected CF | MSE                                  0.0021  0.0033  0.0065   
Projected CF | BPR/random                           0.0056  0.0080  0.0130   
Projected CF | BPR/semantic                         0.0037  0.0048  0.0077   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0037  0.0059  0.0090   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0036  0.0063  0.0094   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0035  0.0059  0.0090   

metric                                                        NDCG          \
k                                                       20      3       5    
model                                                                        
Popularity                                          0.0033  0.0003  0.0003   
BM25                                                0.4413  0.2876  0.3094   
BGE                                                 0.4655  0.2904  0.3136   
Query-KNN CF | MSE                                  0.0126  0.0023  0.0027   
Query-KNN CF | BPR/random                           0.0198  0.0076  0.0091   
Query-KNN CF | BPR/semantic                         0.0174  0.0044  0.0053   
Family CF | MSE                                     0.0124  0.0026  0.0030   
Family CF | BPR/random                              0.0085  0.0018  0.0023   
Family CF | BPR/semantic                            0.0066  0.0006  0.0010   
Projected CF | MSE                                  0.0105  0.0020  0.0026   
Projected CF | BPR/random                           0.0208  0.0056  0.0068   
Projected CF | BPR/semantic                         0.0133  0.0038  0.0044   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0145  0.0033  0.0044   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0148  0.0035  0.0048   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0147  0.0033  0.0045   

metric                                                              \
k                                                       10      20   
model                                                                
Popularity                                          0.0007  0.0014   
BM25                                                0.3369  0.3583   
BGE                                                 0.3452  0.3704   
Query-KNN CF | MSE                                  0.0040  0.0061   
Query-KNN CF | BPR/random                           0.0109  0.0124   
Query-KNN CF | BPR/semantic                         0.0073  0.0094   
Family CF | MSE                                     0.0044  0.0063   
Family CF | BPR/random                              0.0033  0.0044   
Family CF | BPR/semantic                            0.0021  0.0031   
Projected CF | MSE                                  0.0039  0.0052   
Projected CF | BPR/random                           0.0089  0.0115   
Projected CF | BPR/semantic                         0.0056  0.0075   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0057  0.0075   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0061  0.0079   


metric                                              Recall                  \
k                                                       3       5       10   
model                                                                        
Popularity                                          0.0017  0.0021  0.0082   
BM25                                                0.3474  0.4184  0.5145   
BGE                                                 0.3387  0.4121  0.5158   
Query-KNN CF | MSE                                  0.0080  0.0097  0.0116   
Query-KNN CF | BPR/random                           0.0099  0.0121  0.0169   
Query-KNN CF | BPR/semantic                         0.0125  0.0159  0.0244   
Family CF | MSE                                     0.0097  0.0153  0.0235   
Family CF | BPR/random                              0.0075  0.0156  0.0208   
Family CF | BPR/semantic                            0.0057  0.0107  0.0205   
Projected CF | MSE                                  0.0101  0.0119  0.0162   
Projected CF | BPR/random                           0.0143  0.0191  0.0269   
Projected CF | BPR/semantic                         0.0164  0.0218  0.0273   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0064  0.0093  0.0162   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0061  0.0082  0.0139   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0059  0.0086  0.0153   

metric                                                        NDCG          \
k                                                       20      3       5    
model                                                                        
Popularity                                          0.0210  0.0020  0.0022   
BM25                                                0.5981  0.3854  0.4159   
BGE                                                 0.6261  0.3719  0.4042   
Query-KNN CF | MSE                                  0.0209  0.0085  0.0091   
Query-KNN CF | BPR/random                           0.0334  0.0112  0.0119   
Query-KNN CF | BPR/semantic                         0.0410  0.0135  0.0150   
Family CF | MSE                                     0.0316  0.0087  0.0115   
Family CF | BPR/random                              0.0460  0.0070  0.0111   
Family CF | BPR/semantic                            0.0429  0.0056  0.0083   
Projected CF | MSE                                  0.0276  0.0107  0.0113   
Projected CF | BPR/random                           0.0486  0.0135  0.0157   
Projected CF | BPR/semantic                         0.0399  0.0156  0.0182   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0251  0.0066  0.0079   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0273  0.0063  0.0072   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0266  0.0061  0.0072   

metric                                                              \
k                                                       10      20   
model                                                                
Popularity                                          0.0049  0.0091   
BM25                                                0.4580  0.4867   
BGE                                                 0.4494  0.4873   
Query-KNN CF | MSE                                  0.0099  0.0130   
Query-KNN CF | BPR/random                           0.0139  0.0193   
Query-KNN CF | BPR/semantic                         0.0186  0.0244   
Family CF | MSE                                     0.0151  0.0178   
Family CF | BPR/random                              0.0134  0.0219   
Family CF | BPR/semantic                            0.0125  0.0201   
Projected CF | MSE                                  0.0132  0.0171   
Projected CF | BPR/random                           0.0192  0.0264   
Projected CF | BPR/semantic                         0.0207  0.0249   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0108  0.0138   
Family CF | MiniBatchKMeans-128 | count | BPR/r...  0.0096  0.0141   


## Файлы для отчёта

В папке запуска находятся конфигурации и версии зависимостей. В каждой папке `comparison/` лежат общий график `metrics.png`, таблица `metrics.csv`, отдельные таблицы LaTeX для Recall, NDCG и FC при K = 3, 5, 10, 20, доверительные интервалы, предсказания и покрытие. Для вставки LaTeX подключите пакеты `booktabs` и `longtable`.

Папки `validation/` и `family/` содержат таблицы всех допустимых вариантов Family. Таблица `api_mapping.csv` связывает ID в файлах предсказаний с исходными ID группы ToolBench.

## 5. Графики Family CF и сравнения с бейзлайнами

Для ToolLens validation, ToolLens test и ToolBench G1–G3 сохраняем отдельные графики Recall@K, NDCG@K и FC@K со всеми конфигурациями Family CF. На каждом тестовом наборе также строим четыре сравнения с бейзлайнами для Query-KNN, Projected, Family и по одной модели каждого семейства.

Состав последнего сравнения фиксируем по FullCoverage@10, Recall@10 и NDCG@10 на ToolLens validation. Для архивного запуска без валидационных кривых фиксированных моделей график подписывается как описательный, а источник выбора записывается в CSV.

In [6]:
from experiments.reporting import plot_run_results

plot_run_results(run, groups=GROUPS)

,method,model,selection_split,criterion
0,Query-KNN CF,Query-KNN CF | BPR/random,ToolLens test (описательное сравнение),"FullCoverage@10, Recall@10, NDCG@10, model"
1,Projected CF,Projected CF | BPR/random,ToolLens test (описательное сравнение),"FullCoverage@10, Recall@10, NDCG@10, model"
2,Family CF,Family CF | MiniBatchKMeans-128 | count | BPR/...,ToolLens val,"FullCoverage@10, Recall@10, NDCG@10, model"
